In [1]:
from langchain_groq import ChatGroq

In [2]:
llm = ChatGroq(
    temperature=0, 
    groq_api_key='your_api_key', 
    model_name="openai/gpt-oss-120b"
)

response = llm.invoke("The first person to land on moon was ...")
print(response.content)

The first person to set foot on the Moon was **Neil Armstrong**, who did so on July 20 1969 during NASA’s Apollo 11 mission. He famously described the moment as “one small step for [a] man, one giant leap for mankind.”


In [5]:
!pip install beautifulsoup4

  Using cached beautifulsoup4-4.15.0-py3-none-any.whl.metadata (3.8 kB)
Using cached beautifulsoup4-4.15.0-py3-none-any.whl (109 kB)

   ---------------------------------------- 2/2 [beautifulsoup4]



In [31]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.prompts import PromptTemplate
import json

# Step 1: list of individual job URLs (grab these from a listing page, or hardcode for now)
job_urls = [
    "https://careers.nike.com/lead-process-analyst/job/R-88191",
    "https://careers.nike.com/senior-software-engineer-itc/job/R-87481",
    "https://careers.nike.com/supervisor-nike-watertown/job/R-88215",
    "https://careers.nike.com/software-engineer-ii-itc/job/R-77036",
]

prompt_extract = PromptTemplate.from_template(
    """
    ### SCRAPED TEXT FROM WEBSITE:
    {page_data}
    ### INSTRUCTION:
    The scraped text is from the career's page of a website.
    Your job is to extract the job posting and return it in JSON format containing the 
    following keys: `role`, `experience`, `skills` and `description`.
    Only return the valid JSON.
    ### VALID JSON (NO PREAMBLE):    
    """
)

chain_extract = prompt_extract | llm

all_jobs = []

for url in job_urls:
    try:
        loader = WebBaseLoader(url)
        page_data = loader.load().pop().page_content
        res = chain_extract.invoke(input={'page_data': page_data})
        
        # parse the LLM's JSON output into a real Python dict
        job_json = json.loads(res.content)
        all_jobs.append(job_json)
        print(f"✅ Scraped: {url}")
    except Exception as e:
        print(f"❌ Failed on {url}: {e}")

print(json.dumps(all_jobs, indent=2))

✅ Scraped: https://careers.nike.com/lead-process-analyst/job/R-88191
✅ Scraped: https://careers.nike.com/senior-software-engineer-itc/job/R-87481
✅ Scraped: https://careers.nike.com/supervisor-nike-watertown/job/R-88215
✅ Scraped: https://careers.nike.com/software-engineer-ii-itc/job/R-77036
[
  {
    "role": "Lead Process Analyst",
    "experience": "Bachelor\u2019s degree in Computer Science, Supply Chain Management, or Computer Engineering and at least 5 years of progressive post\u2011baccalaureate experience in analysis\u2011related roles.",
    "skills": [
      "SAP",
      "Manhattan WMS",
      "JIRA",
      "Splunk",
      "Agile methodologies",
      "Microfocus ALM",
      "Cognos",
      "SAP Cloud Integration",
      "Predictive analytics",
      "Prescriptive analytics",
      "Statistical/econometric methods",
      "Machine learning techniques",
      "Data extraction, transformation, and modeling",
      "Report development and dashboard creation",
      "SQL/database 

In [32]:
type(res.content)

str

In [33]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()
json_res = json_parser.parse(res.content)
json_res

{'role': 'Software Engineer II, ITC (MFT)',
 'experience': '2-5 years of relevant professional experience; Bachelor’s degree in Computer Science or related field',
 'skills': ['Axway Secure Transport 5.5',
  'Axway Sentinel administration and development',
  'Managed File Transfer (MFT) platform experience',
  'AWS services (EC2, Lambda, S3, RDS, Terraform IaC)',
  'Scripting / programming (e.g., Python, Bash)',
  'Jenkins CI/CD',
  'Git version control',
  'Linux/Unix administration',
  'MySQL / Oracle database administration',
  'Debugging and troubleshooting',
  'Communication with stakeholders',
  'Monitoring and observability tools (New Relic, Splunk)',
  'Automation and self‑service feature development'],
 'description': 'Nike’s Data Integration team (NRTDI) within Foundational Platform Engineering is seeking a Software Engineer II to shape, modernize, and scale the Managed File Transfer (MFT) platform. The role involves developing and maintaining self‑service capabilities for co

In [35]:
!pip install pandas

  Using cached tzdata-2026.3-py2.py3-none-any.whl.metadata (1.4 kB)
   ---------------------------------------- 0.0/10.0 MB ? eta -:--:--
   ---- ----------------------------------- 1.0/10.0 MB 5.0 MB/s eta 0:00:02
   -------- ------------------------------- 2.1/10.0 MB 4.5 MB/s eta 0:00:02
   ----------- ---------------------------- 2.9/10.0 MB 4.3 MB/s eta 0:00:02
   -------------- ------------------------- 3.7/10.0 MB 4.2 MB/s eta 0:00:02
   ----------------- ---------------------- 4.5/10.0 MB 4.1 MB/s eta 0:00:02
   ---------------------- ----------------- 5.5/10.0 MB 4.1 MB/s eta 0:00:02
   ------------------------- -------------- 6.3/10.0 MB 4.1 MB/s eta 0:00:01
   ---------------------------- ----------- 7.1/10.0 MB 4.1 MB/s eta 0:00:01
   ------------------------------- -------- 7.9/10.0 MB 4.1 MB/s eta 0:00:01
   ---------------------------------- ----- 8.7/10.0 MB 4.0 MB/s eta 0:00:01
   -------------------------------------- - 9.7/10.0 MB 4.0 MB/s eta 0:00:01
   ------------

In [36]:
import pandas as pd

df = pd.read_csv("my_portfolio.csv")
df

,Techstack,Links
0,"React, Node.js, MongoDB",https://example.com/react-portfolio
1,"Angular,.NET, SQL Server",https://example.com/angular-portfolio
2,"Vue.js, Ruby on Rails, PostgreSQL",https://example.com/vue-portfolio
3,"Python, Django, MySQL",https://example.com/python-portfolio
4,"Java, Spring Boot, Oracle",https://example.com/java-portfolio
5,"Flutter, Firebase, GraphQL",https://example.com/flutter-portfolio
6,"WordPress, PHP, MySQL",https://example.com/wordpress-portfolio
7,"Magento, PHP, MySQL",https://example.com/magento-portfolio
8,"React Native, Node.js, MongoDB",https://example.com/react-native-portfolio
9,"iOS, Swift, Core Data",https://example.com/ios-portfolio


In [37]:
import uuid
import chromadb

client = chromadb.PersistentClient('vectorstore')
collection = client.get_or_create_collection(name="portfolio")

if not collection.count():
    for _, row in df.iterrows():
        collection.add(documents=row["Techstack"],
                       metadatas={"links": row["Links"]},
                       ids=[str(uuid.uuid4())])

In [42]:
links = collection.query(query_texts=job['skills'], n_results=2).get('metadatas', [])
links

[[{'links': 'https://example.com/ios-ar-portfolio'},
  {'links': 'https://example.com/flutter-portfolio'}],
 [{'links': 'https://example.com/ios-ar-portfolio'},
  {'links': 'https://example.com/angular-portfolio'}],
 [{'links': 'https://example.com/xamarin-portfolio'},
  {'links': 'https://example.com/ml-python-portfolio'}],
 [{'links': 'https://example.com/xamarin-portfolio'},
  {'links': 'https://example.com/ios-ar-portfolio'}],
 [{'links': 'https://example.com/python-portfolio'},
  {'links': 'https://example.com/ml-python-portfolio'}],
 [{'links': 'https://example.com/devops-portfolio'},
  {'links': 'https://example.com/kotlin-backend-portfolio'}],
 [{'links': 'https://example.com/devops-portfolio'},
  {'links': 'https://example.com/kotlin-android-portfolio'}],
 [{'links': 'https://example.com/magento-portfolio'},
  {'links': 'https://example.com/wordpress-portfolio'}],
 [{'links': 'https://example.com/magento-portfolio'},
  {'links': 'https://example.com/wordpress-portfolio'}],
 [{

In [40]:
job = json_res
job['skills']

['Axway Secure Transport 5.5',
 'Axway Sentinel administration and development',
 'Managed File Transfer (MFT) platform experience',
 'AWS services (EC2, Lambda, S3, RDS, Terraform IaC)',
 'Scripting / programming (e.g., Python, Bash)',
 'Jenkins CI/CD',
 'Git version control',
 'Linux/Unix administration',
 'MySQL / Oracle database administration',
 'Debugging and troubleshooting',
 'Communication with stakeholders',
 'Monitoring and observability tools (New Relic, Splunk)',
 'Automation and self‑service feature development']

In [41]:
job

{'role': 'Software Engineer II, ITC (MFT)',
 'experience': '2-5 years of relevant professional experience; Bachelor’s degree in Computer Science or related field',
 'skills': ['Axway Secure Transport 5.5',
  'Axway Sentinel administration and development',
  'Managed File Transfer (MFT) platform experience',
  'AWS services (EC2, Lambda, S3, RDS, Terraform IaC)',
  'Scripting / programming (e.g., Python, Bash)',
  'Jenkins CI/CD',
  'Git version control',
  'Linux/Unix administration',
  'MySQL / Oracle database administration',
  'Debugging and troubleshooting',
  'Communication with stakeholders',
  'Monitoring and observability tools (New Relic, Splunk)',
  'Automation and self‑service feature development'],
 'description': 'Nike’s Data Integration team (NRTDI) within Foundational Platform Engineering is seeking a Software Engineer II to shape, modernize, and scale the Managed File Transfer (MFT) platform. The role involves developing and maintaining self‑service capabilities for co

In [43]:
prompt_email = PromptTemplate.from_template(
        """
        ### JOB DESCRIPTION:
        {job_description}
        
        ### INSTRUCTION:
        You are Mohan, a business development executive at AtliQ. AtliQ is an AI & Software Consulting company dedicated to facilitating
        the seamless integration of business processes through automated tools. 
        Over our experience, we have empowered numerous enterprises with tailored solutions, fostering scalability, 
        process optimization, cost reduction, and heightened overall efficiency. 
        Your job is to write a cold email to the client regarding the job mentioned above describing the capability of AtliQ 
        in fulfilling their needs.
        Also add the most relevant ones from the following links to showcase Atliq's portfolio: {link_list}
        Remember you are Mohan, BDE at AtliQ. 
        Do not provide a preamble.
        ### EMAIL (NO PREAMBLE):
        
        """
        )

chain_email = prompt_email | llm
res = chain_email.invoke({"job_description": str(job), "link_list": links})
print(res.content)

Subject: Accelerating Nike’s MFT Platform Modernization with AtliQ’s Expertise  

Hi [Hiring Manager’s Name],

I’m Mohan, Business Development Executive at AtliQ – an AI‑driven software consulting firm that partners with enterprises to automate and scale critical integration workflows.  

We’ve helped global brands modernize Managed File Transfer (MFT) environments, delivering end‑to‑end self‑service portals, robust Axway Sentinel integrations, and cloud‑native automation on AWS. Our core capabilities align directly with the needs of Nike’s Data Integration team:

- **Axway MFT & Sentinel** – Design, deployment, and ongoing administration of secure transfer pipelines, including custom UI extensions for self‑service configuration.  
- **AWS & IaC** – Automated provisioning of EC2, Lambda, S3, and RDS using Terraform, ensuring repeatable, auditable environments.  
- **DevOps & CI/CD** – Jenkins pipelines, Git‑based version control, and containerized runtimes for rapid feature delivery an